#  Análisis Exploratorio de Burnout en Profesionales de Tecnología

** Dataset:** `dataset_final.csv`  
** Objetivo:** Identificar factores asociados al burnout en perfiles del sector tecnológico.

---

##  1. Análisis Descriptivo

> **¿Qué hacemos aquí?**  
> Cargamos el dataset y realizamos una primera exploración general: cuántos registros tiene, qué variables incluye y cómo lucen las primeras filas. Es el punto de partida de cualquier análisis de datos.

El dataset contiene información de perfiles del sector tech con variables normalizadas entre `0` y `1`, incluyendo factores laborales, hábitos de salud y métricas de desempeño.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Configuración visual global
matplotlib.rcParams.update({
    "figure.facecolor": "#f9f9f9",
    "axes.facecolor": "#f0f0f0",
    "axes.edgecolor": "#cccccc",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "grid.color": "white",
    "grid.linewidth": 1.2,
})

df = pd.read_csv("dataset_final.csv")

print(f" Total de registros : {len(df)}")
print(f" Total de variables : {len(df.columns)}")
print("\n  Columnas disponibles:")
print(df.columns.tolist())
print("\n Primeras filas del dataset:")
df.head()

###  1.1 Tipos de datos y valores nulos

> **¿Qué hacemos aquí?**  
> Verificamos el tipo de cada columna (numérica, texto, etc.) y detectamos si hay valores faltantes. Datos limpios y bien tipados son la base de un análisis confiable.

In [ ]:
print(" Tipos de datos:")
print(df.dtypes)
print("\n Valores nulos por columna:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else " Sin valores nulos")

##  2. Distribución de la Variable Objetivo

> **¿Qué hacemos aquí?**  
> Analizamos cómo se distribuyen los niveles de burnout (`Low`, `Medium`, `High`) en el dataset. Esto nos dice si las clases están balanceadas o si hay sesgo hacia alguna categoría, algo clave antes de construir modelos predictivos.

In [ ]:
if "burnout_level" in df.columns:
    burnout_counts = df["burnout_level"].value_counts(dropna=False)
    print(" Distribución de la variable objetivo:")
    print(burnout_counts)

    porcentaje = (
        df["burnout_level"]
        .value_counts(normalize=True, dropna=False)
        * 100
    )
    print("\n Porcentajes:")
    print(porcentaje.round(2))

##  3. Estadísticas Generales

> **¿Qué hacemos aquí?**  
> Calculamos los promedios de las variables más relevantes del dataset: estrés, calidad de sueño, carga laboral y horas de trabajo diarias. Estos indicadores dan una primera imagen del estado general de la población analizada.

También revisamos la distribución de la variable binaria `high_burnout`, que será nuestra variable objetivo principal para el análisis de correlaciones.

In [ ]:
print(f" Total de registros : {len(df)}")
print(f" Total de variables : {len(df.columns)}")

variables_interes = [
    "stress_level_norm",
    "sleep_quality_norm",
    "workload_index_norm",
    "daily_work_hours_norm"
]

print("\n Promedios de variables clave (escala 0-1):")
for col in variables_interes:
    if col in df.columns:
        print(f"  • {col:<35} {df[col].mean():.3f}")

In [ ]:
if "high_burnout" in df.columns:
    print(" Distribución de High Burnout (0 = No, 1 = Si):")
    print(df["high_burnout"].value_counts())

    porcentaje_high = (
        df["high_burnout"]
        .value_counts(normalize=True)
        * 100
    )
    print("\n Porcentaje:")
    print(porcentaje_high.round(2))

##  4. Detección de Patrones – Correlaciones con `high_burnout`

> **¿Qué hacemos aquí?**  
> Calculamos el coeficiente de correlación de Pearson entre cada variable numérica y el indicador `high_burnout`. Un valor positivo indica que a mayor valor de esa variable, mayor es la probabilidad de burnout elevado; un valor negativo indica efecto protector.

Se identifican las **10 correlaciones más fuertes en cada dirección** para priorizar las variables más relevantes.

In [ ]:
numeric_df = df.select_dtypes(include="number")

if "high_burnout" in numeric_df.columns:
    correlaciones = (
        numeric_df
        .corr()["high_burnout"]
        .sort_values(ascending=False)
    )

    print(" TOP 10 correlaciones POSITIVAS con high_burnout:")
    print(correlaciones.head(10).to_string())

    print("\n TOP 10 correlaciones NEGATIVAS con high_burnout:")
    print(correlaciones.tail(10).to_string())

    correlaciones_df = pd.DataFrame({
        "Variable": correlaciones.index,
        "Correlacion": correlaciones.values
    })
    print("\n Tabla completa de correlaciones:")
    print(correlaciones_df.to_string(index=False))

###  Observaciones relevantes

| Factor | Relacion con burnout |
|--------|---------------------|
|  Nivel de estres | **Fuerte positiva** - principal predictor |
|  Horas de trabajo y pantallas | **Positiva** - mayor tiempo = mayor riesgo |
|  Calidad de sueno | **Negativa** - factor protector clave |
|  Actividad fisica | **Negativa moderada** - efecto protector |
|  Edad y experiencia | **Sin efecto significativo** |

##  5. Visualizacion – Distribucion de Niveles de Burnout

> **¿Qué hacemos aquí?**  
> Representamos visualmente cuantos registros pertenecen a cada categoria de burnout (`Low`, `Medium`, `High`). El grafico de barras permite detectar rapidamente si el dataset esta balanceado o si alguna clase predomina, lo cual influye directamente en la estrategia de modelado.

In [ ]:
if "burnout_level" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor("#f9f9f9")

    counts = df["burnout_level"].value_counts(dropna=False)
    #  Fix: cast index to str para evitar TypeError en el eje categórico
    # de matplotlib cuando el índice contiene floats o NaN (pandas >= 2.x)
    counts.index = counts.index.astype(str)

    color_map = {"High": "#e74c3c", "Medium": "#3498db", "Low": "#2ecc71"}
    colors = [color_map.get(x, "#95a5a6") for x in counts.index]

    bars = ax.bar(counts.index, counts.values, color=colors,
                  edgecolor="white", linewidth=1.5, zorder=3)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 5,
                f"{int(h):,}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")

    ax.set_title("Distribucion de Niveles de Burnout", pad=15)
    ax.set_xlabel("Nivel de Burnout")
    ax.set_ylabel("Cantidad de registros")
    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig("grafico_barras_burnout.png", dpi=150, bbox_inches="tight")
    plt.show()

##  6. Visualizacion – Distribucion del Nivel de Estres

> **¿Qué hacemos aquí?**  
> Graficamos la distribucion de la variable `stress_level_norm` mediante un histograma. Esto nos permite ver si el estres se concentra en valores altos o bajos, si sigue una distribucion normal o si presenta sesgo. Al ser el principal predictor del burnout, entender su distribucion es fundamental.

In [ ]:
if "stress_level_norm" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor("#f9f9f9")

    ax.hist(
        df["stress_level_norm"], bins=10,
        edgecolor="white", linewidth=1.2,
        color="#3498db", zorder=3, alpha=0.9
    )

    media = df["stress_level_norm"].mean()
    ax.axvline(media, color="#e74c3c", linewidth=2,
               linestyle="--", label=f"Media: {media:.3f}", zorder=4)
    ax.legend(fontsize=10)

    ax.set_title("Distribucion del Nivel de Estres (normalizado)", pad=15)
    ax.set_xlabel("Nivel de Estres (0-1)")
    ax.set_ylabel("Frecuencia")
    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig("histograma_estres.png", dpi=150, bbox_inches="tight")
    plt.show()

##  7. Visualizacion – Correlaciones con `high_burnout`

> **¿Qué hacemos aquí?**  
> Mostramos en un grafico de barras horizontales el coeficiente de correlacion de cada variable numerica con `high_burnout`. Las barras rojas representan factores de riesgo (correlacion positiva) y las azules, factores protectores (correlacion negativa). Es una herramienta visual clave para seleccionar variables en modelos de clasificacion.

In [ ]:
if "high_burnout" in numeric_df.columns:
    corr_plot = correlaciones.drop("high_burnout").sort_values()
    colors_corr = ["#e74c3c" if v > 0 else "#3498db" for v in corr_plot]

    fig, ax = plt.subplots(figsize=(9, 11))
    fig.patch.set_facecolor("#f9f9f9")

    ax.barh(corr_plot.index, corr_plot.values,
            color=colors_corr, edgecolor="white",
            linewidth=0.8, zorder=3)

    ax.axvline(0, color="#2c3e50", linewidth=1.2,
               linestyle="--", zorder=4)
    ax.set_title("Correlacion de Variables con High Burnout", pad=15)
    ax.set_xlabel("Coeficiente de Correlacion de Pearson")
    ax.xaxis.grid(True, zorder=0, alpha=0.7)
    ax.set_axisbelow(True)

    from matplotlib.patches import Patch
    legend = [
        Patch(color="#e74c3c", label="Factor de riesgo (+)"),
        Patch(color="#3498db", label="Factor protector (-)")
    ]
    ax.legend(handles=legend, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.savefig("correlaciones_burnout.png", dpi=150, bbox_inches="tight")
    plt.show()

##  8. Conclusiones

> **¿Qué hacemos aquí?**  
> Sintetizamos los hallazgos mas importantes del analisis exploratorio, identificando que factores tienen mayor impacto sobre el burnout y que implicancias tienen para futuras estrategias de prevencion o modelado.

---

###  Hallazgos principales

1. ** El estres es el principal predictor**: presenta la correlacion mas fuerte con `high_burnout` entre todas las variables analizadas.
2. ** Jornadas extensas aumentan el riesgo**: tanto las horas de trabajo como el tiempo frente a pantallas se asocian positivamente con el burnout elevado.
3. ** La calidad del sueno es el factor protector mas potente**: los profesionales con mejor descanso reportan niveles significativamente menores de agotamiento.
4. ** La actividad fisica tiene un efecto protector moderado**: aunque menor que el sueno, contribuye a reducir el riesgo.
5. ** Edad y experiencia no explican el burnout**: su influencia en este dataset es practicamente nula.

---

###  Conclusion general

El dataset muestra que el burnout esta mas asociado a **condiciones de trabajo y bienestar diario** que a caracteristicas personales o de trayectoria profesional. Esto respalda la hipotesis de que la **sobrecarga laboral y la falta de recuperacion fisica** son los factores clave en la aparicion del agotamiento.

Futuras estrategias de prevencion deberian enfocarse en la **gestion del estres**, el **equilibrio trabajo-descanso** y la **promocion de habitos saludables**.